# Analyze SWE Predictions

The final notebook! Here, I will perform analysis of MACA SWE patterns and compare to the PRISM baseline.

## Preliminary: Set up notebook

In [ ]:
# import libraries

# file management
import os
import pathlib
from pathlib import Path
# import pyarrow.parquet as pq

# datatypes
import numpy as np
import pandas as pd
import xarray as xr

# geospatial data
import geopandas as gpd
import rioxarray as rxr

# plotting
import holoviews as hv
import hvplot.pandas
import hvplot.xarray
import matplotlib
import matplotlib.pyplot as plt
import panel as pn

# misc
import math

In [ ]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')
output_dir = Path(proj_dir, 'outputs')

In [ ]:
# import datasets

# prism pred swe
prism_swe_path = 

# prism climate
prism_climate_path = Path(cleaned_data_dir, 'prism', 'prism_mhw_1990_2020_cleaned_with_crs.nc')
prism_clim_ds = xr.read_netcdf(prism_climate_path)

# maca pred swe
models = {'CNRM-CM5', 'GFDL-ESM2M', 'CanESM2', 'IPSL-CM5A-MR'}

# maca climate

## Question 1: How will the Spatial Distribution of SWE change in the Missouri Headwaters watershed?

This step will analyze how the spatial distribution of SWE will change, both across the spatial extent of the watershed and across elevation.

#### 1a: Absolute and maybe Anomaly plots

Have to make sure the grids line up for anomaly plots.

In [ ]:
# prism plots

# get mean of prism swe
p_mean_swe_ds = 

# make mean plot

# save plot

# show plot

In [ ]:
# maca plots

# get cbar range from prism plots?

# make 5 subplots

# set names of models including MMM
maca_plot_names = {'CNRM-CM5', 'GFDL-ESM2M', 'CanESM2', 'IPSL-CM5A-MR', 'Multi-Model Mean'}

# calculate MMM

# loop through models, make plots

# save panel

# show panel

#### 1b: Hovmöller Diagram

## Question 2: How will the magnitude and timing of peak SWE change in the future?

Make sure to total SWE across the basin before mean!

#### 2a: Basin volume hydrograph

Use average PRISM basin volume as normal, four ensemble plots and MMM (use all years)

In [ ]:
# set up basin vol dict
maca_basin_vol = {}

# loop through, calculate basin vol

# convert swe to km and multiply by grid cell size
pred_ds['pred_swe_km3'] = pred_ds['swe_pred'] * 1e-6 * 16

# get daily basin volume
daily_vol = pred_ds['pred_swe_km3'].sum(dim=['latitude', 'longitude'])

# get max SWE volume and date per water year
ann_peak_vol = (daily_vol.groupby('water_year').max(dim = 'date'))
peak_vol_date = daily_vol.groupby('water_year').map(lambda x: x.idxmax(dim = 'date'))

# combine into df
peak_swe_df = pd.DataFrame({
    'peak_swe_km3': ann_peak_vol.to_pandas(),
    'peak_date': peak_vol_date.to_pandas()
}).reset_index()



# compare day of year
peak_swe_df['doy'] = peak_swe_df['peak_date'].dt.day_of_year

peak_swe_df

#### 2b: Histogram and/or box-and-whisker plots

Box and whisker plots:
This will show monthly SWE for PRISM and MACA (not sure if 4 models or multi-model mean), and I expect it will demonstrate a shift earlier in the year, with decreased magnitude.

Histogram:
Showing True DOY for peak SWE for PRISM and MACA. Finer look at the timing shift.

#### 2c: Line plots, possibly with dual axis

WY on x-axis, True Peak SWE and April 1 SWE on y-axis. This will show the magnitude of peak SWE and how it differs from the traditional 'April 1' benchmark. Probably just use MMM here.

If there can be a secondary y axis with DOY for peak SWE, that would be cool. But it might be easier to just rely on the histograms for that.

## Question 3: How is the climatology of the Missouri Headwaters changing? Are snow droughts - warm or cold - becoming more frequent?

This will determine baseline climatology from PRISM, and then assess how climate in the MACA timefrmae changes. The goal is to start to understand the prevalence of snow drought; however, strict criteria of 'snow drought' may be lacking.

#### 3a: Precip vs. SWE scatter

Precip on x-axis, SWE on y-axis; color scatters by average temperature. Average Precip/SWE line on the 45. This should demonstrate if there are more winters with normal or high precip, but low SWE. Likely will see a clustering of red points in the bottom right corner.

#### 3b: Diverging annual anomaly bar chart

Average SWE from PRISM is the normal. Bars go above and below normal, sized on SWE, and are colored by average temp or degree days.

## Step 5: Mean SWE animations

#### Step 5a: PRISM Animation

#### Step 5b: MACA animations

Maybe just do an ensemble mean here?

## Step 6: Basin volume hydrographs

Other ideas:

Rain-to-Snow Fraction & Freezing Line Shifts: Since your model relies on temperature arrays, you can analyze the raw meteorological inputs to show why the SWE is changing. Calculate the percentage of winter precipitation falling as rain versus snow (using a simple $0^\circ\text{C}$ threshold) or track the average altitude of the winter freezing line in PRISM versus MACA.

"True Peak" vs. April 1st Disconnect: Historically, water managers rely on April 1st as the standard metric for peak SWE. You can plot the $\Delta$SWE between the "True Peak" date and April 1st for each year. If this gap widens significantly in the MACA projections, you have a compelling operational conclusion: April 1st is becoming an obsolete metric for water forecasting in the MHW basin.

Peak SWE Timing Shifts ($\Delta$DOY): You previously found a nearly one-month shift in peak SWE timing across your historical baseline. You can formalize this by plotting the Day of Water Year (DOY) of the basin's maximum SWE volume for the historical period versus the mid-century ensemble. A histogram or boxplot comparing the distribution of these peak dates will clearly quantify how much earlier reservoirs will fill (and potentially spill) under the MACA scenarios.